In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid

In [ ]:
models = {
    "bn": GGNBGB,
    "binomial": GGBinomialGB,
    "bernoulli": GGBinomialGB,
    "poisson": GGPoissonGB,
}

df = (
    pd.read_csv("../../ggkm/data/melanoma.csv")
    .assign(status=lambda x: (x["status"] == 1).astype(int))
    .fillna(0)
    .rename(columns={"status": "delta"})
)

for model_name, model_class in models.items():
    results = (
        pd.read_csv(f"../../ggkm/data/melanoma_results/boost/melanoma_{model_name}_boost.csv")
        .assign(
            test_ibs=lambda x: x.test_ibs.apply(parse_cell),
            test_cindex=lambda x: x.test_cindex.apply(parse_cell),
            test_auc=lambda x: x.test_auc.apply(parse_cell),
            best_params=lambda x: x.best_params.apply(parse_cell),
        )
        .explode(["test_ibs", "test_cindex", "best_params"])
        .rename(columns={"test_cindex": "cindex"})
    )

    plot_survival_curves_gb(
        df, results, model_class=model_class,
        preprocessor=MelanomaSurvivalPreprocessor(),
        distribution_name=model_name,
        test_size=0.4
    )

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))

ax.hlines(4, 0, 10, lw=4)

ax.plot(7, 4, "o", markersize=18)

ax.text(
    7,
    4.25,
    "Event",
    ha="center",
    fontsize=15
)

ax.hlines(3, 0, 10, lw=4)

ax.hlines(
    3,
    10,
    12,
    lw=2.5,
    linestyles="dashed"
)

ax.plot(
    12,
    3,
    ">",
    markersize=18
)

ax.text(
    10.2,
    3.15,
    "Event occurs\nafter observation",
    ha="left",
    fontsize=15
)

ax.hlines(2, 0, 10, lw=4)

ax.hlines(
    2,
    -2,
    0,
    lw=2.5,
    linestyles="dashed"
)

ax.plot(
    -2,
    2,
    "<",
    markersize=18
)

ax.text(
    -0.2,
    2.15,
    "Event occurred\nbefore observation",
    ha="right",
    fontsize=15
)

ax.hlines(1, 0, 10, lw=4)

ax.plot(
    [4, 7],
    [1, 1],
    "|",
    markersize=18,
    mew=2
)

ax.fill_between(
    [4, 7],
    0.85,
    1.15,
    alpha=0.3
)

ax.text(
    5.5,
    1.25,
    "Event occurred\nwithin this interval",
    ha="center",
    fontsize=13
)

for y in [1, 2, 3, 4]:
    ax.plot(
        0,
        y,
        "ks",
        markersize=18
    )

    ax.plot(
        10,
        y,
        "ks",
        markersize=18
    )

ax.text(
    0,
    4.6,
    "Start",
    ha="center",
    fontsize=15
)

ax.text(
    10,
    4.6,
    "End",
    ha="center",
    fontsize=15
)

ax.set_yticks([1, 2, 3, 4])

ax.set_yticklabels(
    [
        "Interval censored",
        "Left censored",
        "Right censored",
        "Observed event"
    ],
    fontsize=15
)

ax.set_xticks([])

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlim(-3, 13)
ax.set_ylim(0.5, 5)

ax.set_xlabel("")
ax.set_title(
    "",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
t = np.linspace(0.01, 10, 1000)

def hazard_high(t):
    return (
        0.9 * np.exp(-3.5 * t)
        + 0.22 * np.exp(-0.12 * t)
        + 0.015
    )

def hazard_low(t):
    return (
        0.015
        + 0.005 * t
        + 0.002 * np.exp(2.0 * (t - 7))
    )

h_high = hazard_high(t)
h_low = hazard_low(t)

H_high = cumulative_trapezoid(h_high, t, initial=0)
H_low = cumulative_trapezoid(h_low, t, initial=0)

S_high = np.exp(-H_high)
S_low = np.exp(-H_low)

fig, axes = plt.subplots(
    2, 2,
    figsize=(9, 6),
    constrained_layout=True
)

ax = axes[0, 0]

ax.plot(t, h_high, linewidth=2)

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Hazard", fontsize=12)

ax.text(
    0.5, 1.05, "a",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=13,
    fontweight="bold"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlim(0, 10)
ax.set_ylim(bottom=0)

ax = axes[1, 0]

ax.plot(t, S_high, linewidth=2)

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Survival Probability", fontsize=12)

ax.text(
    0.5, 1.05, "b",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=13,
    fontweight="bold"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlim(0, 10)
ax.set_ylim(0, 1.02)

ax.set_yticks([0, 1])

ax = axes[0, 1]

ax.plot(t, h_low, linewidth=2)

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Hazard", fontsize=12)

ax.text(
    0.5, 1.05, "c",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=13,
    fontweight="bold"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlim(0, 10)
ax.set_ylim(bottom=0)


ax = axes[1, 1]

ax.plot(t, S_low, linewidth=2)

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Survival Probability", fontsize=12)

ax.text(
    0.5, 1.05, "d",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=13,
    fontweight="bold"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_xlim(0, 10)
ax.set_ylim(0, 1.02)

ax.set_yticks([0, 1])

plt.show()